In [1]:
"""
====================================================
ERA5 Visualization Script
====================================================
"""

'\n====================================================\nERA5 Visualization Script\n====================================================\n'

In [2]:
#######################
#DIRECTORIES

In [3]:
# #SETTING UP DIRECTORIES
# mainDirectory = '/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/'
# workingDirectory="/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/DataAnalysis/InputData_DataAnalysis/"
# print(workingDirectory)
# outputDirectory=workingDirectory+"OUTPUT/"
# dataDirectory=mainDirectory+"DownloadData/DATA/ERA5_Data/"

In [4]:
#SETTING UP DIRECOTRIES
mainDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/"
codeDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/"
def SetOutputDirectory(campaign):
    import os
    outputDirectory=codeDirectory+"OUTPUT/DataAnalysis/ERA5_Data"
    outputDirectory=os.path.join(outputDirectory, campaign)
    os.makedirs(outputDirectory, exist_ok=True)
    return outputDirectory

def SetDataDirectory(campaign):
    import os
    dataDirectory=codeDirectory+"DATA/ERA5_Data/"
    dataDirectory=os.path.join(dataDirectory, campaign)
    os.makedirs(dataDirectory, exist_ok=True)
    return dataDirectory

In [5]:
#######################
#LIBRARIES, FUNCTIONS, and CLASSES

In [6]:
#IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Libraries/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [7]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [8]:
#IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/Classes/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Classes_InputData_DataAnalysis",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [9]:
###########################
#FUNCTIONS

In [10]:
#MAKE DATE FOLDER (for output) FUNCTION
def MakeDateFolder(date_string):
    date_folder = strings.DateString(date_string)
    #adding date to output folder
    subdir = os.path.join(outputDirectory, date_folder)
    os.makedirs(subdir, exist_ok=True)
    return date_folder

def GetLoadDirectory(dataDirectory,date_folder):
    loadDirectorys = [
        os.path.join(dataDirectory, date_folder, f"{var}_ERA5_{date_folder}.nc")
        for var in variables.keys()
    ]
    return loadDirectorys

In [11]:
#RUN CALCULATIONS and PLOTTING #*#*#*#*#*#*# (this version adds quiver to U/V plots)
def RunCalculations(numerics, var_data, units, variable, calculation, mult_factor):
    calculation_results = {}

    arr   = var_data
    units = units
    if mult_factor != "NaN":
        arr *= mult_factor

    t, _ = Ultimate_AreaAverage(var_data, dims=('t','y','x'), dim_names=('t',), mode='keep')

    three_hours = 3 * numerics.hour_index
    tyx_3h = calculation.block_vertical_profiles_3D(arr, block=three_hours)

    calculation_results[variable] = {
        "units": units,
        "t": t,            # (t,)
        "tyx_3h": tyx_3h # (nblocks, y, x)
    }

    return calculation_results

def RunPlots(numerics, calculation_results, date_string, UTC_offset, outputFile, plotting, 
             colormap, vline, data_lim, line_contour, center_contour):
    for name, result in calculation_results.items():
        common_args = {
            "var_name": name,
            "var_units": result["units"],
            "date_string": date_string,
            "date_folder":  date_folder,
            "outputFile": outputFile,
            "numerics": numerics,
            "data_lim": data_lim,
            "UTC_offset": UTC_offset,
        }

        plotting.TimeSeries(var_data=result['t'], **common_args)
        plotting.MultiAverage_HorizontalFields_Surface(var_data=result['tyx_3h'], plev=1000, line_contour=line_contour, colormap=colormap, center_contour=center_contour, **common_args)

#RUNNING CALCULATIONS FUNCTION
def RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting):
    # global variable,calculation_results_temp1,calculation_results_temp2 #*#* quiver
    for count, (loadDirectory, (variable, components)) in enumerate(tqdm(zip(loadDirectorys, variables.items()), total=len(loadDirectorys), desc="Running Calculations"),start=1):
        units, colormap, vline, mult_factor = components["unit"], components["colormap"], components["vline"], components["mult_factor"]
        data_lim, line_contour, center_contour = components["data_lim"], components["line_contour"], components["center_contour"]
        
        #print
        print(f"Plotting {len(variables)} Variables",'\n')
        print(f"{count}. {variable} ({units}) → {loadDirectory}")
        
        #loading the variable
        ncFile=xr.open_dataset(loadDirectory)
        var_name = [v for v in list(ncFile.data_vars) if v not in ["number", "expver"]][0]
        # print('\n',var_name,'***')
        var_data=ncFile[var_name].data
        numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,
                           TIME=ncFile[var_name]['valid_time'].data,P=[1000], LAT=ncFile[var_name]['latitude'].data,LON=ncFile[var_name]['longitude'].data)
        print(variable+":\n","\t(Nt, Nlat, Nlon) = ",(numerics.Nt,numerics.Nlat,numerics.Nlon),"\n")
    
        #making output filename
        outputFile = os.path.join(outputDirectory, date_folder, variable) #variable also can be var_name
        
        os.makedirs(outputFile, exist_ok=True)
    
        #doing calculations
        calculation_results=RunCalculations(numerics, var_data, "("+units+")", variable, calculation, mult_factor)
    
        #plotting
        RunPlots(numerics, calculation_results, date_string, UTC_offset, outputFile, plotting, 
                 colormap, vline, data_lim, line_contour, center_contour)

In [12]:
###########################
#LOADING DATA

In [13]:
#load in ERA5 data
variables = {
    "convective_available_potential_energy": {"unit": r"$J\ kg^{-1}$", "colormap": "YlOrRd", "vline": "NaN", "mult_factor": "NaN", "data_lim": "NaN", "line_contour": "F", "center_contour": "NaN",},
    "convective_inhibition": {"unit": r"$J\ kg^{-1}$", "colormap": "Blues", "vline": "NaN", "mult_factor": "NaN", "data_lim": "NaN", "line_contour": "F", "center_contour": "NaN",},
}

In [14]:
###########################
#RUNNING

In [ ]:
##########################################################
# DOWNLOADING TRACER CAMPAIGN DATA
##########################################################

In [15]:
# coorindates information
# Data BOUNDING BOX centered at Houston, TX Mobile Facility (TRACER) Facility S2 ==> CSAP (C-Band Scanning ARM Precipitation Radar)
# (29.532N, 95.284W)
outputDirectory=SetOutputDirectory(campaign="TRACER")
dataDirectory=SetDataDirectory(campaign="TRACER")
#UTC OFFSET
UTC_offset="-5"

In [16]:
###########################
#DATE ONE (DRY CASE)

#date information
date_string = "06-08 - 06-10 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/1 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_44563/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,
/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/Functions_2.0/AreaAverageFunctions.py:77: RuntimeWarning: Mean of empty slice
  out = np.nanmean(data, axis=tuple(axes))


Plotting 1 Variables 

1. convective_inhibition ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-08_-_06-10_2022/convective_inhibition_ERA5_06-08_-_06-10_2022.nc
convective_inhibition:
 	(Nt, Nlat, Nlon) =  (72, 20, 23) 



Running Calculations: 100%|██████████| 1/1 [00:14<00:00, 14.62s/it]


In [19]:
###########################
#DATE TWO (MOIST CASE)

#date information
date_string = "06-30 - 07-02 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/1 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_44563/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,
/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/Functions_2.0/AreaAverageFunctions.py:77: RuntimeWarning: Mean of empty slice
  out = np.nanmean(data, axis=tuple(axes))


Plotting 1 Variables 

1. convective_inhibition ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-30_-_07-02_2022/convective_inhibition_ERA5_06-30_-_07-02_2022.nc
convective_inhibition:
 	(Nt, Nlat, Nlon) =  (72, 20, 23) 



Running Calculations: 100%|██████████| 1/1 [00:07<00:00,  7.35s/it]


In [20]:
###########################
#DATE THREE (INTERESTING CASE)

#date information
date_string = "08-11 - 08-13 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/1 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_44563/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,
/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/Functions_2.0/AreaAverageFunctions.py:77: RuntimeWarning: Mean of empty slice
  out = np.nanmean(data, axis=tuple(axes))


Plotting 1 Variables 

1. convective_inhibition ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/08-11_-_08-13_2022/convective_inhibition_ERA5_08-11_-_08-13_2022.nc
convective_inhibition:
 	(Nt, Nlat, Nlon) =  (72, 20, 23) 



Running Calculations: 100%|██████████| 1/1 [00:07<00:00,  7.27s/it]


In [70]:
######################################

In [14]:
##########################################################
# DOWNLOADING PRECIP CAMPAIGN DATA
##########################################################

In [15]:
# coorindates information
# Data BOUNDING BOX centered at Hsinchu, Taiwan PRECIP Campaign S-Pol radar moments data collected during the Prediction of Rainfall Extremes Campaign In the Pacific (PRECIP)
# (24.82N, 120.91E)
outputDirectory=SetOutputDirectory(campaign="PRECIP")
dataDirectory=SetDataDirectory(campaign="PRECIP")
#UTC OFFSET
UTC_offset="+8"

In [16]:
###########################
#DATE ONE (DRY CASE)

#date information
date_string = "06-05 - 06-07 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/2 [00:00<?, ?it/s]

Plotting 2 Variables 

1. convective_available_potential_energy ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/06-05_-_06-07_2022/convective_available_potential_energy_ERA5_06-05_-_06-07_2022.nc


/glade/derecho/scratch/aroseman/tmp/ipykernel_60550/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


convective_available_potential_energy:
 	(Nt, Nlat, Nlon) =  (72, 20, 22) 



Running Calculations:  50%|█████     | 1/2 [00:16<00:16, 16.50s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_60550/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,
/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/Functions_2.0/AreaAverageFunctions.py:77: RuntimeWarning: Mean of empty slice
  out = np.nanmean(data, axis=tuple(axes))


Plotting 2 Variables 

2. convective_inhibition ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/06-05_-_06-07_2022/convective_inhibition_ERA5_06-05_-_06-07_2022.nc
convective_inhibition:
 	(Nt, Nlat, Nlon) =  (72, 20, 22) 



Running Calculations: 100%|██████████| 2/2 [00:24<00:00, 12.29s/it]


In [17]:
###########################
#DATE TWO (MOIST CASE)

#date information
date_string = "07-16 - 07-18 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/2 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_60550/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 2 Variables 

1. convective_available_potential_energy ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/07-16_-_07-18_2022/convective_available_potential_energy_ERA5_07-16_-_07-18_2022.nc
convective_available_potential_energy:
 	(Nt, Nlat, Nlon) =  (72, 20, 22) 



Running Calculations:  50%|█████     | 1/2 [00:08<00:08,  8.45s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_60550/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,
/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/Functions_2.0/AreaAverageFunctions.py:77: RuntimeWarning: Mean of empty slice
  out = np.nanmean(data, axis=tuple(axes))


Plotting 2 Variables 

2. convective_inhibition ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/07-16_-_07-18_2022/convective_inhibition_ERA5_07-16_-_07-18_2022.nc
convective_inhibition:
 	(Nt, Nlat, Nlon) =  (72, 20, 22) 



Running Calculations: 100%|██████████| 2/2 [00:16<00:00,  8.38s/it]


In [ ]:
#####################################################

In [ ]:
#####################################################

In [56]:
#*#* Some Possible Future Improvements #*#*
#1. All Plots: x- and y-ticks somewhat incomplete